In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import random

# Configuración del dispositivo (GPU si está disponible para acelerar el entrenamiento)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

In [ ]:
# Transformación base: convertir a tensor y normalizar entre 0 y 1
transform = transforms.Compose([
    transforms.ToTensor()
])

# Descargar datasets de entrenamiento y prueba
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64,
                                         shuffle=False, num_workers=2)

# Clases del dataset
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
def add_gaussian_noise(imgs, sigma=0.1):
    """
    Tarea 1: Ruido Gaussiano.
    Agrega una perturbación aleatoria N(0, sigma)[cite: 62, 63].
    """
    noise = torch.randn_like(imgs) * sigma
    noisy_imgs = imgs + noise
    return torch.clamp(noisy_imgs, 0., 1.)

def add_salt_and_pepper_noise(imgs, amount=0.05):
    """
    Tarea 1: Ruido Sal y Pimienta.
    Reemplaza píxeles por 0 (pimienta) o 1 (sal)[cite: 68, 72, 73].
    """
    noisy_imgs = imgs.clone()
    batch_size, channels, h, w = noisy_imgs.size()
    
    for i in range(batch_size):
        # Máscaras de probabilidad
        prob = torch.rand(h, w)
        salt = prob < (amount / 2)
        pepper = (prob > (amount / 2)) & (prob < amount)
        
        for c in range(channels):
            noisy_imgs[i, c, salt] = 1.0
            noisy_imgs[i, c, pepper] = 0.0
            
    return noisy_imgs

def convert_to_grayscale(imgs):
    """
    Tarea 2: Colorización.
    Convierte a escala de grises[cite: 90]. 
    Repetimos el canal 3 veces para mantener la arquitectura del modelo de 3 canales de entrada[cite: 38].
    """
    grayscale = transforms.Grayscale(num_output_channels=3)(imgs)
    return grayscale

def apply_inpainting_mask(imgs, mask_size=12):
    """
    Tarea 3: Inpainting.
    Reemplaza una región cuadrada por valores cero[cite: 108, 109].
    """
    masked_imgs = imgs.clone()
    h, w = imgs.shape[2], imgs.shape[3]
    
    # Coordenadas aleatorias para el cuadrado negro
    x = random.randint(0, w - mask_size)
    y = random.randint(0, h - mask_size)
    
    masked_imgs[:, :, y:y+mask_size, x:x+mask_size] = 0.0
    return masked_imgs

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=64):
        super(ConvAutoencoder, self).__init__()
        
        # Encoder [cite: 39, 40]
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),  # Salida: 16x16
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), # Salida: 8x8
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), # Salida: 4x4
            nn.ReLU(),
            nn.Flatten()
        )
        
        # Proyección al Latent Space [cite: 41, 42]
        self.fc_encode = nn.Linear(64 * 4 * 4, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 64 * 4 * 4)
        
        # Decoder [cite: 43, 44]
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (64, 4, 4)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), # Salida: 8x8
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), # Salida: 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(16, 3, 3, stride=2, padding=1, output_padding=1),  # Salida: 32x32
            nn.Sigmoid() # Para mantener los valores entre 0 y 1
        )

    def forward(self, x):
        x = self.encoder(x)
        latent = self.fc_encode(x)
        x = self.fc_decode(latent)
        x = self.decoder(x)
        return x

# Instanciar el modelo
model = ConvAutoencoder(latent_dim=64).to(device)
print(model)

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_autoencoder(model, dataloader, epochs=10, task='denoising_gaussian'):
    model.train()       
    train_losses = []

    for epoch in range(epochs):
        running_loss = 0.0
        for data in dataloader:
            imgs, _ = data
            targets = imgs.to(device) # La imagen objetivo siempre es la original [cite: 56, 86, 104]
            
            # Generar la entrada dependiendo de la tarea
            if task == 'denoising_gaussian':
                inputs = add_gaussian_noise(imgs, sigma=0.1) # [cite: 65]
            elif task == 'denoising_sp':
                inputs = add_salt_and_pepper_noise(imgs)
            elif task == 'colorization':
                inputs = convert_to_grayscale(imgs) # [cite: 84]
            elif task == 'inpainting':
                inputs = apply_inpainting_mask(imgs, mask_size=12) # [cite: 122]
            
            inputs = inputs.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(inputs)
            
            # Calcular pérdida y optimizar
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        epoch_loss = running_loss / len(dataloader)
        train_losses.append(epoch_loss)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")
        
    return train_losses


In [ ]:
def visualize_results(model, dataloader, task='denoising_gaussian', num_imgs=5):
    model.eval()
    dataiter = iter(dataloader)
    imgs, _ = next(dataiter)
    
    # Generar entrada según la tarea
    if task == 'denoising_gaussian':
        inputs = add_gaussian_noise(imgs, sigma=0.1)
    elif task == 'denoising_sp':
        inputs = add_salt_and_pepper_noise(imgs)
    elif task == 'colorization':
        inputs = convert_to_grayscale(imgs)
    elif task == 'inpainting':
        inputs = apply_inpainting_mask(imgs, mask_size=12)
        
    inputs = inputs.to(device)
    with torch.no_grad():
        outputs = model(inputs)
        
    inputs = inputs.cpu()
    outputs = outputs.cpu()
    
    # Graficar
    fig, axes = plt.subplots(3, num_imgs, figsize=(15, 6))
    for i in range(num_imgs):
        # Original
        axes[0, i].imshow(np.transpose(imgs[i].numpy(), (1, 2, 0)))
        axes[0, i].axis('off')
        if i == 0: axes[0, i].set_title("Original", fontsize=14)
        
        # Entrada (Corrompida/Gris/Enmascarada)
        axes[1, i].imshow(np.transpose(inputs[i].numpy(), (1, 2, 0)))
        axes[1, i].axis('off')
        if i == 0: axes[1, i].set_title("Input Modelo", fontsize=14)
        
        # Reconstruida
        axes[2, i].imshow(np.transpose(outputs[i].numpy(), (1, 2, 0)))
        axes[2, i].axis('off')
        if i == 0: axes[2, i].set_title("Reconstruida", fontsize=14)
        
    plt.tight_layout()
    plt.show()


In [ ]:
# --- TAREA 1: Denoising ---
print("Iniciando entrenamiento para Denoising...")
model_denoising = ConvAutoencoder(latent_dim=64).to(device) # Instancia nueva
optimizer_denoising = optim.Adam(model_denoising.parameters(), lr=0.001) # Optimizador nuevo

losses_den = train_autoencoder(model_denoising, trainloader, epochs=10, task='denoising_gaussian')
visualize_results(model_denoising, testloader, task='denoising_gaussian')


# --- TAREA 2: Colorization ---
print("Iniciando entrenamiento para Colorization...")
model_color = ConvAutoencoder(latent_dim=64).to(device) # Instancia nueva
optimizer_color = optim.Adam(model_color.parameters(), lr=0.001)

# Nota: tendrías que modificar ligeramente tu función train_autoencoder para que 
# reciba el optimizador como parámetro, o asegurarte de que use el global correcto.
# Si tu función usa el 'optimizer' global de la celda de arriba, lo mejor es reasignarlo:
optimizer = optimizer_color 
losses_col = train_autoencoder(model_color, trainloader, epochs=10, task='colorization')
visualize_results(model_color, testloader, task='colorization')


# --- TAREA 3: Inpainting ---
print("Iniciando entrenamiento para Inpainting...")
model_inpainting = ConvAutoencoder(latent_dim=64).to(device) # Instancia nueva
optimizer_inpainting = optim.Adam(model_inpainting.parameters(), lr=0.001)

optimizer = optimizer_inpainting
losses_inp = train_autoencoder(model_inpainting, trainloader, epochs=10, task='inpainting')
visualize_results(model_inpainting, testloader, task='inpainting')